<a href="https://colab.research.google.com/github/ryussy/aiplun-ambassador/blob/main/%E5%88%86%E6%95%A3%E8%A1%A8%E7%8F%BE_ipynb_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習：単語・文をベクトルで表す 3つの方法を比べよう

**自然言語処理 第9回「分散表現」 — プログラミング演習**

---
## この演習の目的
同じ小さなコーパスを **3つの方法** でベクトル化し、それぞれの違いを **自分のコードと数値で** 体験します。

1. **bag-of-words（BoW）** … 文書を単語の出現回数で表す（復習）
2. **カウントベース** … 共起行列 ＋ PPMI で「単語」を表す
3. **推論ベース（Word2Vec）** … 予測で単語ベクトルを学習する

最後に、3手法を表で比較し、考察します。

## 進め方
- 各パートには「**例**（実行して動きを確認）」と「**課題**（自分で書く）」があります。
- `# TODO` のセルにコードを書いて実行してください。


## 0. 準備（ライブラリの読み込み）
必要なら次のセルのコメントを外してインストールしてください。

In [ ]:
!pip install gensim   # 実行してパッケージをインストール

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.5 MB/s eta 0:00:00


In [ ]:
import numpy as np

# 共通コーパス（すでに分かち書き済み。日本語は本来 MeCab 等で分割します）
corpus = [
    ["犬", "が", "公園", "で", "走る"],
    ["猫", "が", "公園", "で", "走る"],
    ["犬", "が", "餌", "を", "食べる"],
    ["猫", "が", "餌", "を", "食べる"],
    ["私", "は", "魚", "を", "食べる"],
]

# 語彙（vocabulary）と単語→ID の辞書を作る
vocab = sorted({w for s in corpus for w in s})
word2id = {w: i for i, w in enumerate(vocab)}
print('語彙数:', len(vocab))
print('語彙:', vocab)

語彙数: 11
語彙: ['が', 'は', 'を', '公園', '犬', '猫', '私', '走る', '食べる', '餌', '魚']


## Part A — bag-of-words（文書ベクトル）
BoW は **文書** を、単語の出現回数のベクトルで表します。**語順は無視**されます。

In [ ]:
# 【例】1文を BoW ベクトルにする関数
def bow_vector(sentence, word2id):
    v = np.zeros(len(word2id), dtype=int)
    for w in sentence:
        if w in word2id:
            v[word2id[w]] += 1
    return v

# 文① と 文② を BoW にして表示
print('語彙   :', vocab)
print('文①   :', bow_vector(corpus[0], word2id))
print('文②   :', bow_vector(corpus[1], word2id))

語彙   : ['が', 'は', 'を', '公園', '犬', '猫', '私', '走る', '食べる', '餌', '魚']
文①   : [1 0 1 1 1 0 0 1 0 0 0]
文②   : [1 0 1 1 0 1 0 1 0 0 0]


### 課題 A-1：語順は保たれる？
次の2文を BoW にして、ベクトルが等しいか確認してください。

- `sent_A = "猫 が 魚 を 食べる".split()`
- `sent_B = "魚 が 猫 を 食べる".split()`（主語と目的語が逆）

In [ ]:
sent_A = "猫 が 魚 を 食べる".split()
sent_B = "魚 が 猫 を 食べる".split()

In [ ]:
# TODO: sent_A と sent_B の BoW を作り、np.array_equal で比較してみよう
# ヒント: bow_vector(sent_A, word2id) など

### 課題 A-2（記述）
BoW 表現で、単語「犬」と「猫」の**意味の近さ**を測ることはできますか？ できない場合、その理由を「次元」という言葉を使って説明してください。

➡ ここに回答を書いてください：

## Part B — カウントベース（共起行列 → PPMI）
今度は **単語** を、「周りにどの語が出るか（共起）」で表します。分散仮説どおり、似た文脈に出る語は似たベクトルになるはずです。

In [ ]:
# 【例】共起行列を作る（window = 前後の語数）
def co_matrix(corpus, word2id, window=1):
    V = len(word2id)
    M = np.zeros((V, V), dtype=int)
    for s in corpus:
        ids = [word2id[w] for w in s]
        for i, wi in enumerate(ids):
            for j in range(max(0, i-window), min(len(ids), i+window+1)):
                if i != j:
                    M[wi][ids[j]] += 1
    return M

# コサイン類似度
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

M = co_matrix(corpus, word2id, window=1)
print('sim(犬, 猫) =', round(cosine(M[word2id['犬']], M[word2id['猫']]), 3))
print('sim(犬, 魚) =', round(cosine(M[word2id['犬']], M[word2id['魚']]), 3))

sim(犬, 猫) = 1.0
sim(犬, 魚) = 0.0


「犬」と「猫」は同じ文脈（が・公園・餌・走る・食べる）に出るので似た値、
「犬」と「魚」は文脈が違うので低い値になったはずです。これが**分散仮説**です。

In [ ]:
# 【例】PPMI（高頻度語の影響を補正する）
def ppmi(M):
    M = M.astype(float)
    total = M.sum()
    row = M.sum(axis=1, keepdims=True)   # P(w)
    col = M.sum(axis=0, keepdims=True)   # P(c)
    P  = M / total
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log2(P / ((row/total) * (col/total)))
    pmi[~np.isfinite(pmi)] = 0.0
    return np.maximum(pmi, 0.0)

W = ppmi(M)
print('PPMI後 sim(犬, 猫) =', round(cosine(W[word2id['犬']], W[word2id['猫']]), 3))

PPMI後 sim(犬, 猫) = 1.0


### 課題 B-1：ウィンドウサイズを変える
`window=2` で共起行列を作り直し、`sim(犬, 猫)` と `sim(犬, 魚)` がどう変わるか調べてください。

In [ ]:
# TODO: window=2 で co_matrix を作り、犬-猫 と 犬-魚 のコサイン類似度を比較しよう


### 課題 B-2（記述）
BoW（Part A）と共起行列（Part B）の **いちばん大きな違い** は何ですか？ 「何を表しているか（文書 / 単語）」に注目して書いてください。

➡ ここに回答を書いてください：

## Part C — 推論ベース（Word2Vec）
最後に、予測でベクトルを学習する Word2Vec を使います。

> 注意：このコーパスはとても小さいので結果は不安定です。傾向の確認が目的です。本格的に試したい人は、後半の**学習済みモデル**を使ってください。

In [ ]:
from gensim.models import Word2Vec

# 【例】小さなコーパスで学習（sg=1: Skip-gram, 0: CBOW）
model = Word2Vec(corpus, vector_size=50, window=2, min_count=1, sg=1, epochs=300, seed=1)

print('猫 に近い語:', model.wv.most_similar('猫', topn=3))
print('sim(犬, 猫) =', round(model.wv.similarity('犬', '猫'), 3))

猫 に近い語: [('食べる', 0.22970058023929596), ('が', 0.18802212178707123), ('犬', 0.17898572981357574)]
sim(犬, 猫) = 0.179


### 課題 C-1：学習済みモデルでアナロジー
大きな学習済みモデルを読み込み、類似語とアナロジーを試してください（初回はダウンロードに時間がかかります）。

In [ ]:
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-50')        # 英語・約66MB（軽い）

print(wv.most_similar('cat', topn=5))


[==================================================] 100.0% 66.0/66.0MB downloaded
[('dog', 0.9218006134033203), ('rabbit', 0.8487821221351624), ('monkey', 0.8041081428527832), ('rat', 0.7891963124275208), ('cats', 0.7865270972251892)]


In [ ]:
# # TODO: アナロジー  king - man + woman ≒ ?  を計算してみよう
# # ヒント: wv.most_similar(positive=[...], negative=[...])

### 課題 C-2（記述）
共起行列（Part B）と比べて、Word2Vec の **利点** と、その代わりに **必要になるもの** を1つずつ挙げてください。

➡ ここに回答を書いてください：

## Part D (ボーナス点） — 3手法の比較と考察

### 課題 D-1：比較表を埋めよう
これまでの結果をもとに、表の空欄（？）を埋めてください。

| 観点 | bag-of-words | カウントベース（共起+PPMI） | Word2Vec |
|---|---|---|---|
| 表す対象（文書 / 単語） | ？ | ？ | ？ |
| 語順 | ？ | ？ | ？ |
| ベクトル（疎・高次元 / 密・低次元） | ？ | ？ | ？ |
| 単語どうしの類似度が測れるか | ？ | ？ | ？ |
| 作り方（数える / 予測して学習） | ？ | ？ | ？ |
| データ追加・大規模化のしやすさ | ？ | ？ | ？ |

### 課題 D-2：考察（記述）
1. 同じ「犬 ≈ 猫」を、なぜ BoW では表せず、共起行列と Word2Vec では表せるのでしょうか。
2. それでも BoW が今も使われる場面はどんなときだと思いますか。
3. 共起行列に対する Word2Vec の利点は何ですか。
4. 3手法に共通する弱点は何ですか（ヒント：多義語）。次回扱う「文脈化表現」がなぜ必要か、関連づけて述べてください。

➡ ここに回答を書いてください：